# 01c Prepare RegGS Scene (405841, Full, 512)

Goal:

1. adapt `405841` into one RegGS-ready full scene;
2. use Part1 `images_512` as image source (symlink, no copy);
3. use FRONT calib + gt_old pose to export `intrinsics.json` and `cameras.json`;
4. keep file layout aligned with Re10k-1 / DL3DV-2 prepared scenes;
5. run compatibility checks against RegGS loader requirements.

Canonical prepared-data path:

```text
dataset/405841/part2/reggs_405841_full_512/
```

In [1]:
from pathlib import Path
import csv
import json
import math
import os
import shutil
import struct
import numpy as np

CV_ROOT = Path('~/CV_Project').expanduser()
DATASET_ROOT = CV_ROOT / 'dataset'

SOURCE_ROOT = DATASET_ROOT / '405841'
SOURCE_FRONT_ROOT = SOURCE_ROOT / 'FRONT'
SOURCE_IMAGE_DIR_512 = SOURCE_ROOT / 'part1' / 'shared' / 'images_512'
SOURCE_IMAGE_DIR_RAW = SOURCE_FRONT_ROOT / 'rgb'
SOURCE_CALIB_DIR = SOURCE_FRONT_ROOT / 'calib'
SOURCE_GT_DIR = SOURCE_FRONT_ROOT / 'gt'
SOURCE_GT_OLD_DIR = SOURCE_FRONT_ROOT / 'gt_old'
PART2_ROOT = SOURCE_ROOT / 'part2'

PART2_ROOT.mkdir(parents=True, exist_ok=True)

print('SOURCE_IMAGE_DIR_512 =', SOURCE_IMAGE_DIR_512)
print('SOURCE_IMAGE_DIR_RAW =', SOURCE_IMAGE_DIR_RAW)
print('SOURCE_CALIB_DIR =', SOURCE_CALIB_DIR)
print('SOURCE_GT_OLD_DIR =', SOURCE_GT_OLD_DIR)
print('PART2_ROOT =', PART2_ROOT)

SOURCE_IMAGE_DIR_512 = /home/bzhang512/CV_Project/dataset/405841/part1/shared/images_512
SOURCE_IMAGE_DIR_RAW = /home/bzhang512/CV_Project/dataset/405841/FRONT/rgb
SOURCE_CALIB_DIR = /home/bzhang512/CV_Project/dataset/405841/FRONT/calib
SOURCE_GT_OLD_DIR = /home/bzhang512/CV_Project/dataset/405841/FRONT/gt_old
PART2_ROOT = /home/bzhang512/CV_Project/dataset/405841/part2


## 1. Export configuration

In [2]:
EXPORT_SCENE_NAME = 'reggs_405841_full_512'
EXPORT_ROOT = PART2_ROOT / EXPORT_SCENE_NAME

POSE_SOURCE = 'gt_old'  # choices: 'gt_old', 'gt'
OVERWRITE_EXPORT = True

SPARSE_STRIDE = 10
MIN_TRAIN_VIEWS = 3
INCLUDE_LAST_FRAME_IN_TRAIN = True

print('EXPORT_ROOT =', EXPORT_ROOT)
print('POSE_SOURCE =', POSE_SOURCE)
print('OVERWRITE_EXPORT =', OVERWRITE_EXPORT)
print('SPARSE_STRIDE =', SPARSE_STRIDE)
print('MIN_TRAIN_VIEWS =', MIN_TRAIN_VIEWS)
print('INCLUDE_LAST_FRAME_IN_TRAIN =', INCLUDE_LAST_FRAME_IN_TRAIN)

EXPORT_ROOT = /home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_full_512
POSE_SOURCE = gt_old
OVERWRITE_EXPORT = True
SPARSE_STRIDE = 10
MIN_TRAIN_VIEWS = 3
INCLUDE_LAST_FRAME_IN_TRAIN = True


## 2. Helpers

In [3]:
def read_png_size(path: Path):
    png_sig = bytes.fromhex('89504E470D0A1A0A')
    with path.open('rb') as f:
        sig = f.read(8)
        if sig != png_sig:
            raise ValueError(f'Not a PNG file: {path}')
        _length = struct.unpack('>I', f.read(4))[0]
        chunk_type = f.read(4)
        if chunk_type != b'IHDR':
            raise ValueError(f'Invalid PNG header: {path}')
        width, height = struct.unpack('>II', f.read(8))
    return width, height


def safe_reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def symlink_file(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    os.symlink(src, dst)


def parse_calib_intrinsics(calib_path: Path):
    text = calib_path.read_text(encoding='utf-8').strip().splitlines()
    if len(text) < 2:
        raise ValueError(f'Invalid calib file (too short): {calib_path}')

    vals = {}
    parts = text[1].replace(':', '').split()
    for i in range(0, len(parts) - 1, 2):
        k = parts[i]
        v = parts[i + 1]
        if k in {'fx', 'fy', 'cx', 'cy'}:
            vals[k] = float(v)

    if any(k not in vals for k in ('fx', 'fy', 'cx', 'cy')):
        raise ValueError(f'Cannot parse intrinsics from: {calib_path}')
    return vals['fx'], vals['fy'], vals['cx'], vals['cy']


def read_pose_4x4(path: Path):
    lines = [ln.strip() for ln in path.read_text(encoding='utf-8').splitlines() if ln.strip()]
    if len(lines) < 4:
        raise ValueError(f'Pose file must have 4 rows: {path}')
    rows = []
    for i in range(4):
        vals = [float(x) for x in lines[i].split()]
        if len(vals) != 4:
            raise ValueError(f'Pose row {i} in {path} does not have 4 columns')
        rows.append(vals)
    return np.array(rows, dtype=float)


def rotation_to_quat_xyzw(R: np.ndarray):
    m00, m01, m02 = R[0, 0], R[0, 1], R[0, 2]
    m10, m11, m12 = R[1, 0], R[1, 1], R[1, 2]
    m20, m21, m22 = R[2, 0], R[2, 1], R[2, 2]
    tr = m00 + m11 + m22

    if tr > 0.0:
        s = math.sqrt(tr + 1.0) * 2.0
        qw = 0.25 * s
        qx = (m21 - m12) / s
        qy = (m02 - m20) / s
        qz = (m10 - m01) / s
    elif (m00 > m11) and (m00 > m22):
        s = math.sqrt(1.0 + m00 - m11 - m22) * 2.0
        qw = (m21 - m12) / s
        qx = 0.25 * s
        qy = (m01 + m10) / s
        qz = (m02 + m20) / s
    elif m11 > m22:
        s = math.sqrt(1.0 + m11 - m00 - m22) * 2.0
        qw = (m02 - m20) / s
        qx = (m01 + m10) / s
        qy = 0.25 * s
        qz = (m12 + m21) / s
    else:
        s = math.sqrt(1.0 + m22 - m00 - m11) * 2.0
        qw = (m10 - m01) / s
        qx = (m02 + m20) / s
        qy = (m12 + m21) / s
        qz = 0.25 * s

    q = np.array([qx, qy, qz, qw], dtype=float)
    q = q / np.linalg.norm(q)
    return q.tolist()


def choose_train_test_local_ids(n_frames: int, stride: int, min_train: int, include_last: bool):
    if n_frames <= 0:
        return [], []

    train_ids = list(range(0, n_frames, stride))
    if include_last:
        train_ids.append(n_frames - 1)
    train_ids = sorted(set(train_ids))

    if len(train_ids) < min_train:
        extra = np.linspace(0, n_frames - 1, min_train).astype(int).tolist()
        train_ids = sorted(set(train_ids + extra))

    test_ids = [i for i in range(n_frames) if i not in set(train_ids)]
    return train_ids, test_ids


def write_json(path: Path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding='utf-8')

## 3. Load 405841 source (images_512 + FRONT calib/pose)

In [4]:
def load_405841_source_full_512():
    image_paths = sorted(SOURCE_IMAGE_DIR_512.glob('*.png'), key=lambda p: p.stem)
    calib_paths = sorted(SOURCE_CALIB_DIR.glob('*.txt'), key=lambda p: p.stem)

    if POSE_SOURCE == 'gt_old':
        pose_dir = SOURCE_GT_OLD_DIR
    elif POSE_SOURCE == 'gt':
        pose_dir = SOURCE_GT_DIR
    else:
        raise ValueError(f'Unsupported POSE_SOURCE: {POSE_SOURCE}')

    pose_paths = sorted(pose_dir.glob('*.txt'), key=lambda p: p.stem)

    if not image_paths:
        raise FileNotFoundError(f'No images found in {SOURCE_IMAGE_DIR_512}')
    if len(image_paths) != len(calib_paths) or len(image_paths) != len(pose_paths):
        raise ValueError(
            f'Count mismatch: images={len(image_paths)}, calib={len(calib_paths)}, poses={len(pose_paths)}'
        )

    export_width, export_height = read_png_size(image_paths[0])
    raw_ref_path = SOURCE_IMAGE_DIR_RAW / image_paths[0].name
    raw_width, raw_height = read_png_size(raw_ref_path)

    fx, fy, cx, cy = parse_calib_intrinsics(calib_paths[0])
    intrinsics = {
        'fx': fx / raw_width,
        'fy': fy / raw_height,
        'cx': cx / raw_width,
        'cy': cy / raw_height,
    }

    for cal in calib_paths[1:]:
        _fx, _fy, _cx, _cy = parse_calib_intrinsics(cal)
        if not (
            abs(_fx - fx) < 1e-9
            and abs(_fy - fy) < 1e-9
            and abs(_cx - cx) < 1e-9
            and abs(_cy - cy) < 1e-9
        ):
            raise ValueError(f'Per-frame intrinsics are not constant: {cal.name}')

    records = []
    for i, (img, cal, pose) in enumerate(zip(image_paths, calib_paths, pose_paths)):
        if img.stem != cal.stem or img.stem != pose.stem:
            raise ValueError(f'Stem mismatch at index {i}: {img.name}, {cal.name}, {pose.name}')
        records.append({
            'global_idx': i,
            'stem': img.stem,
            'image_path': img,
            'calib_path': cal,
            'pose_path': pose,
        })

    return {
        'records': records,
        'export_width': export_width,
        'export_height': export_height,
        'calib_ref_width': raw_width,
        'calib_ref_height': raw_height,
        'intrinsics': intrinsics,
        'pose_dir': pose_dir,
    }


source = load_405841_source_full_512()
print('num frames =', len(source['records']))
print('export size =', (source['export_width'], source['export_height']))
print('calib reference size =', (source['calib_ref_width'], source['calib_ref_height']))
print('normalized intrinsics =', source['intrinsics'])
print('pose dir =', source['pose_dir'])

num frames = 199
export size = (512, 512)
calib reference size = (1920, 1280)
normalized intrinsics = {'fx': 1.0764049814673433, 'fy': 1.6146074722010149, 'cx': 0.4950787903203501, 'cy': 0.5009273860525132}
pose dir = /home/bzhang512/CV_Project/dataset/405841/FRONT/gt_old


## 4. Export one full scene

In [5]:
def make_export_camera(local_idx: int, global_idx: int, pose_w2c: np.ndarray, intrinsics: dict):
    r = pose_w2c[:3, :3]
    t = pose_w2c[:3, 3]
    quat_xyzw = rotation_to_quat_xyzw(r)
    return {
        'cam_id': int(local_idx),
        'cam_quat': [float(x) for x in quat_xyzw],
        'cam_trans': [float(x) for x in t.tolist()],
        'cx': float(intrinsics['cx']),
        'cy': float(intrinsics['cy']),
        'fx': float(intrinsics['fx']),
        'fy': float(intrinsics['fy']),
        'image_name': f'{local_idx:04d}.png',
        'timestamp': int(global_idx),
    }


def export_full_scene(source_dict: dict, export_root: Path):
    if export_root.exists() and not OVERWRITE_EXPORT:
        raise FileExistsError(f'{export_root} already exists and OVERWRITE_EXPORT=False')

    safe_reset_dir(export_root)
    image_dir = export_root / 'images'
    image_dir.mkdir(parents=True, exist_ok=True)

    records = source_dict['records']
    export_cameras = []
    index_map = []

    for local_idx, rec in enumerate(records):
        dst = image_dir / f'{local_idx:04d}.png'
        symlink_file(rec['image_path'], dst)

        pose = read_pose_4x4(rec['pose_path'])
        export_cameras.append(
            make_export_camera(local_idx, rec['global_idx'], pose, source_dict['intrinsics'])
        )

        index_map.append({
            'local_idx': int(local_idx),
            'global_idx': int(rec['global_idx']),
            'local_image_name': f'{local_idx:04d}.png',
            'source_image_name': rec['image_path'].name,
            'source_pose_file': rec['pose_path'].name,
            'source_calib_file': rec['calib_path'].name,
        })

    write_json(export_root / 'intrinsics.json', source_dict['intrinsics'])
    write_json(export_root / 'cameras.json', export_cameras)
    write_json(export_root / 'index_map.json', index_map)

    train_ids_local, test_ids_local = choose_train_test_local_ids(
        n_frames=len(records),
        stride=SPARSE_STRIDE,
        min_train=MIN_TRAIN_VIEWS,
        include_last=INCLUDE_LAST_FRAME_IN_TRAIN,
    )
    split_payload = {
        'scene_name': EXPORT_SCENE_NAME,
        'num_frames': int(len(records)),
        'sparse_stride': int(SPARSE_STRIDE),
        'min_train_views': int(MIN_TRAIN_VIEWS),
        'include_last_frame_in_train': bool(INCLUDE_LAST_FRAME_IN_TRAIN),
        'train_ids_local': [int(x) for x in train_ids_local],
        'test_ids_local': [int(x) for x in test_ids_local],
    }
    write_json(export_root / 'split_train_test.json', split_payload)

    summary = {
        'scene_name': EXPORT_SCENE_NAME,
        'export_root': str(export_root),
        'num_frames': int(len(records)),
        'num_train': int(len(train_ids_local)),
        'num_test': int(len(test_ids_local)),
        'export_width': int(source_dict['export_width']),
        'export_height': int(source_dict['export_height']),
        'calib_ref_width': int(source_dict['calib_ref_width']),
        'calib_ref_height': int(source_dict['calib_ref_height']),
        'use_symlinks': True,
    }
    return summary


scene_summary = export_full_scene(source, EXPORT_ROOT)
print(json.dumps(scene_summary, indent=2))

{
  "scene_name": "reggs_405841_full_512",
  "export_root": "/home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_full_512",
  "num_frames": 199,
  "num_train": 21,
  "num_test": 178,
  "export_width": 512,
  "export_height": 512,
  "calib_ref_width": 1920,
  "calib_ref_height": 1280,
  "use_symlinks": true
}


## 5. Verify export and RegGS compatibility

In [6]:
export_images = sorted((EXPORT_ROOT / 'images').iterdir())
export_cameras = json.loads((EXPORT_ROOT / 'cameras.json').read_text(encoding='utf-8'))
export_intrinsics = json.loads((EXPORT_ROOT / 'intrinsics.json').read_text(encoding='utf-8'))
split_payload = json.loads((EXPORT_ROOT / 'split_train_test.json').read_text(encoding='utf-8'))

assert len(export_images) == len(export_cameras) == scene_summary['num_frames']
assert export_images[0].name == '0000.png'
assert export_cameras[0]['image_name'] == '0000.png'
assert export_cameras[0]['cam_id'] == 0

w0, h0 = read_png_size(export_images[0])
assert (w0, h0) == (scene_summary['export_width'], scene_summary['export_height'])

required_intrinsics_keys = {'fx', 'fy', 'cx', 'cy'}
required_camera_keys = {'cam_id', 'cam_quat', 'cam_trans', 'cx', 'cy', 'fx', 'fy', 'image_name'}
assert set(export_intrinsics.keys()) == required_intrinsics_keys
assert required_camera_keys.issubset(set(export_cameras[0].keys()))

train_ids = split_payload['train_ids_local']
test_ids = split_payload['test_ids_local']
assert len(set(train_ids).intersection(set(test_ids))) == 0
assert sorted(train_ids + test_ids) == list(range(scene_summary['num_frames']))

ref_scenes = [
    CV_ROOT / 'dataset' / 'Re10k-1' / 'part2' / 'reggs_re10k1_fullseq_256',
    CV_ROOT / 'dataset' / 'DL3DV-2' / 'part2' / 'reggs_dl3dv2_fullseq_256',
]
for ref in ref_scenes:
    assert (ref / 'images').exists(), f'Missing images in {ref}'
    assert (ref / 'intrinsics.json').exists(), f'Missing intrinsics.json in {ref}'
    assert (ref / 'cameras.json').exists(), f'Missing cameras.json in {ref}'
    ref_intr = json.loads((ref / 'intrinsics.json').read_text(encoding='utf-8'))
    ref_cam = json.loads((ref / 'cameras.json').read_text(encoding='utf-8'))
    assert set(ref_intr.keys()) == required_intrinsics_keys
    assert required_camera_keys.issubset(set(ref_cam[0].keys()))

print('Export structure and RegGS compatibility checks passed.')
print('export root =', EXPORT_ROOT)
print('num images =', len(export_images))
print('intrinsics =', export_intrinsics)
print('first camera keys =', sorted(export_cameras[0].keys()))

Export structure and RegGS compatibility checks passed.
export root = /home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_full_512
num images = 199
intrinsics = {'fx': 1.0764049814673433, 'fy': 1.6146074722010149, 'cx': 0.4950787903203501, 'cy': 0.5009273860525132}
first camera keys = ['cam_id', 'cam_quat', 'cam_trans', 'cx', 'cy', 'fx', 'fy', 'image_name', 'timestamp']


## 6. Write summary artifacts

In [7]:
scene_summary_json = EXPORT_ROOT / 'scene_summary.json'
scene_summary_csv = EXPORT_ROOT / 'scene_summary.csv'
manifest_path = EXPORT_ROOT / 'scene_manifest.json'

write_json(scene_summary_json, [scene_summary])
with scene_summary_csv.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=[
        'scene_name',
        'export_root',
        'num_frames',
        'num_train',
        'num_test',
        'export_width',
        'export_height',
        'calib_ref_width',
        'calib_ref_height',
        'use_symlinks',
    ])
    writer.writeheader()
    writer.writerow({k: scene_summary[k] for k in writer.fieldnames})

manifest = {
    'source_scene': '405841',
    'export_scene_name': EXPORT_SCENE_NAME,
    'pose_source': POSE_SOURCE,
    'image_source': str(SOURCE_IMAGE_DIR_512),
    'calib_source': str(SOURCE_CALIB_DIR),
    'pose_source_dir': str(source['pose_dir']),
    'export_size': {
        'width': int(source['export_width']),
        'height': int(source['export_height']),
    },
    'calib_reference_size': {
        'width': int(source['calib_ref_width']),
        'height': int(source['calib_ref_height']),
    },
    'resize_applied_in_this_notebook': False,
    'intrinsics_convention': 'normalized_xy_by_calib_reference_size',
    'reggs_layout': ['images/', 'intrinsics.json', 'cameras.json'],
}
write_json(manifest_path, manifest)

print('wrote:', scene_summary_json)
print('wrote:', scene_summary_csv)
print('wrote:', manifest_path)
print('Done: 405841 full 512 RegGS scene is prepared.')

wrote: /home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_full_512/scene_summary.json
wrote: /home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_full_512/scene_summary.csv
wrote: /home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_full_512/scene_manifest.json
Done: 405841 full 512 RegGS scene is prepared.
